In [12]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
import xml.etree.ElementTree as ET
from sklearn.multioutput import MultiOutputClassifier

# Define paths to folders
folders = []

# Annotation types we are interested in
annotation_types = ["ObstructiveApnea", "CentralApnea", "MixedApnea", "Hypopnea", "Snore"]

# Function to parse annotations from .rml file
def parse_annotations(rml_file_path):
    namespace = '{http://www.respironics.com/PatientStudy.xsd}'
    tree = ET.parse(rml_file_path)
    root = tree.getroot()
    annotations = []
    events_section = root.find(f"{namespace}ScoringData/{namespace}Events")
    if events_section is not None:
        for event in events_section.findall(f"{namespace}Event"):
            event_type = event.attrib.get("Type", "")
            start_time = float(event.attrib.get("Start", "0"))
            duration = float(event.attrib.get("Duration", "0"))
            if event_type in annotation_types:
                annotations.append({
                    'start': start_time,
                    'end': start_time + duration,
                    'type': event_type
                })
    return annotations

# Function to extract features (MFCC) from audio aligned with annotations
def extract_features(wav_file_path, annotations, sr=22050):
    features = []
    labels = []
    try:
        audio, _ = librosa.load(wav_file_path, sr=sr)
        audio_length = len(audio) / sr
        print(f"Loaded full audio file: {wav_file_path} with duration {audio_length} seconds")
        
        for ann in annotations:
            start_frame = int(ann['start'] * sr)
            end_frame = int(ann['end'] * sr)
            
            # Skip if the annotation is out of audio bounds
            if end_frame > len(audio):
                print(f"Skipping annotation {ann['type']} (start: {ann['start']}, end: {ann['end']}) - out of bounds for {wav_file_path}")
                continue
            
            segment = audio[start_frame:end_frame]
            
            # Extract MFCC features
            mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
            mfcc_mean = np.mean(mfcc.T, axis=0)
            
            features.append(mfcc_mean)
            labels.append(ann['type'])
            
    except Exception as e:
        print(f"Error processing file {wav_file_path}: {e}")
        
    return np.array(features), labels

# Load data and annotations
all_features = []
all_labels = []
for folder in folders:
    wav_files = [f for f in os.listdir(folder) if f.endswith('.wav')]
    rml_files = [f for f in os.listdir(folder) if f.endswith('.rml')]
    
    if not wav_files:
        print(f"No .wav files found in folder {folder}")
    if not rml_files:
        print(f"No .rml files found in folder {folder}")
    
    # Assuming there's only one .rml file per folder
    rml_file_path = os.path.join(folder, rml_files[0])
    annotations = parse_annotations(rml_file_path)
    
    if not annotations:
        print(f"No relevant annotations found in {rml_file_path}")
        continue
    
    # Process each .wav file in the folder using the same annotations
    for wav_file in wav_files:
        wav_file_path = os.path.join(folder, wav_file)
        print(f"Using annotations from {rml_file_path} for .wav file: {wav_file_path}")
        
        # Extract features from audio with annotations
        features, labels = extract_features(wav_file_path, annotations)
        if features.size > 0:
            all_features.extend(features)
            all_labels.extend(labels)
        else:
            print(f"No features extracted from {wav_file_path}")

# Check if we have any data to proceed
if len(all_features) == 0 or len(all_labels) == 0:
    print("No data available for training. Check the file paths and ensure that annotations exist in the .rml files.")
else:
    # Convert labels to multi-label format
    mlb = MultiLabelBinarizer(classes=annotation_types)
    y = mlb.fit_transform([[label] for label in all_labels])

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(all_features, y, test_size=0.2, random_state=42)

    # Train model
    model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
    model.fit(X_train, y_train)

    # Evaluate model
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=mlb.classes_))


Using annotations from /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/filtered_00001057-100507.rml for .wav file: /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/00001057-100507_snore_cleaned.wav
Loaded full audio file: /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/00001057-100507_snore_cleaned.wav with duration 12802.0 seconds
Using annotations from /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/filtered_00001057-100507.rml for .wav file: /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01

/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
